In [2]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [3]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [4]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [5]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [6]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.item_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Task 2: Xử lý NULL, Xử lý Outlier

## Xử lý null

### Xử lý age_group

In [7]:
df_age = read_parquet_item("./preprocessed-dataset")
df_age.head()

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new,gender_target_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Không xác định""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định""","""Bé Gái""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Không xác định""","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""[""Từ 6M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""12-36M"""


#### Ý tưởng 3:

Thống kê

In [8]:
import polars as pl

df = df_age   # hoặc df_filled tùy bạn đang dùng

# Hàm tiện dụng để lấy danh sách unique của 1 cột
def get_unique_list(df, col):
    return (
        df.select(col)
          .unique()
          .sort(col)
          .get_column(col)
          .to_list()
    )

# Lấy tất cả class
age_list    = get_unique_list(df, "age_group_final")

# Gom vào dictionary
category_dict = {
    "age_group": age_list
}

print(len(age_list))

# In ra theo từng dòng, rất dễ đọc
print("\n=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===\n")
for key, value in category_dict.items():
    print(f"{key}: {value}\n")

308

=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===

age_group: ['0-10M', '0-11M', '0-120M', '0-12M', '0-144M', '0-14M', '0-18M', '0-1M', '0-24M', '0-2M', '0-30M', '0-36M', '0-3M', '0-48M', '0-4M', '0-5M', '0-60M', '0-6M', '0-72M', '0-84M', '0-9M', '1-12M', '1-18M', '1-24M', '1-36M', '1-3M', '108-120M', '108-132M', '12-108M', '12-120M', '12-132M', '12-144M', '12-14M', '12-192M', '12-24M', '12-36M', '12-48M', '12-60M', '12-72M', '120-144M', '12M-18M', '12M-4Y', '13-16M', '13-17M', '132-144M', '13M-24M', '14-17M', '18-24M', '18M-24M', '18M-36M', '18M-4Y', '1M-12M', '1M-15M', '1M-3M', '24-120M', '24-36M', '24-48M', '24-60M', '24-72M', '24-96M', '2M-15M', '2M-6M', '3-18M', '3-24M', '3-6M', '36-120M', '36-144M', '36-216M', '36-48M', '36-60M', '36-72M', '3M-12M', '3M-18M', '3M-24M', '3M-6M', '4-24M', '4-30M', '4-6M', '48-60M', '48-72M', '4M-4Y', '4M-6M', '6-12M', '6-144M', '6-15M', '6-18M', '6-24M', '6-30M', '6-36M', '6-60M', '6-72M', '6-84M', '6-8M', '6-9M', '60-72M', '6M-10M', '6M-12M', 